# GS CitizenTech · Vésper — Pipeline Big Data
**Disciplina:** Big Data · **Produto:** Vésper (ruas escuras — proposta) · **Lakehouse:** Databricks Free Edition

Arquitetura: `Fontes (SP156 · Open-Meteo · IBGE) → Airflow → chegada/ → Volume → Bronze → Prata → Ouro → Painel`

**Escopo Vésper:** a base aberta do SP156 (3º tri/2026) não tem serviço de lâmpada apagada. Usamos os chamados de **poste e fiação** como indicador da infraestrutura de iluminação.

**Ordem:** rode de cima para baixo. Pré-requisito: arquivos dos lotes no Volume `/Volumes/workspace/gs/chegada` (soltos na raiz).

In [0]:
%pip install openpyxl -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# openpyxl p/ ler xlsx: o %pip acima instala; aqui reinicia o Python para carregar
dbutils.library.restartPython()

In [0]:
# CONFIG — escopo do Vésper e caminhos (rode antes de tudo)
BASE   = "/Volumes/workspace/gs/chegada"
SCHEMA = "workspace.gs"

# SP156 não tem "lâmpada apagada": usamos poste + fiação como indicador de iluminação
ESCOPO_SERVICOS = [
    "Denunciar Fiação solta na calçada",
    "Denunciar Poste em mau estado",
]
print("escopo:", ESCOPO_SERVICOS)

escopo: ['Denunciar Fiação solta na calçada', 'Denunciar Poste em mau estado']


## Bronze — dado bruto + rastro (`arquivo_origem`, `carregado_em`)
As 3 fontes entram por **COPY INTO** (incremental e idempotente: o COPY INTO guarda quais arquivos já carregou). O COPY INTO não lê `.xlsx`, então cada planilha nova do SP156 vira um `.csv` de pouso — só muda o formato; o conteúdo continua bruto e todo em texto.

In [0]:
# BRONZE — xlsx -> csv de pouso (idempotente) + COPY INTO incremental nas 3 fontes
import glob, os, unicodedata
import pandas as pd
from pyspark.sql import functions as F

POUSO_SP156 = f"{BASE}/_pouso_sp156"

def _limpa(c):
    c = unicodedata.normalize("NFKD", str(c)).encode("ascii", "ignore").decode()
    return c.strip().lower().replace(" ", "_")

def converter_sp156():
    os.makedirs(POUSO_SP156, exist_ok=True)
    novos = 0
    for xlsx in sorted(glob.glob(f"{BASE}/sp156_*.xlsx")):
        csv = f"{POUSO_SP156}/{os.path.basename(xlsx).replace('.xlsx', '.csv')}"
        if os.path.exists(csv):
            continue                                   # já convertido
        df = pd.read_excel(xlsx, sheet_name="Solicitações", dtype=str)
        df.columns = [_limpa(c) for c in df.columns]
        df.to_csv(csv, index=False)
        novos += 1
    print("xlsx convertidos agora:", novos)

def _copy_into(tabela, origem, padrao, formato, opcoes):
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SCHEMA}.{tabela}")
    spark.sql(f"""
      COPY INTO {SCHEMA}.{tabela}
      FROM (SELECT *, _metadata.file_name AS arquivo_origem,
                   current_timestamp() AS carregado_em
            FROM '{origem}')
      FILEFORMAT = {formato} PATTERN = '{padrao}'
      FORMAT_OPTIONS ({opcoes})
      COPY_OPTIONS ('mergeSchema' = 'true')""")

def carregar_bronze():
    converter_sp156()
    _copy_into("bronze_sp156", POUSO_SP156, "sp156_*.csv", "CSV",
               "'header'='true', 'multiLine'='true', 'escape'='\"'")
    _copy_into("bronze_clima", BASE, "clima_*.json", "JSON", "'multiLine'='true'")
    _copy_into("bronze_ibge", BASE, "municipios_sp_*.json", "JSON", "'multiLine'='true'")
    for t in ["bronze_sp156", "bronze_clima", "bronze_ibge"]:
        print(t, spark.table(f"{SCHEMA}.{t}").count())

def registrar(etapa):
    linhas = []
    for t in ["bronze_sp156", "bronze_clima", "bronze_ibge"]:
        df = spark.table(f"{SCHEMA}.{t}")
        linhas.append((etapa, t, df.count(), df.select("arquivo_origem").distinct().count()))
    (spark.createDataFrame(linhas, "etapa string, tabela string, linhas long, arquivos long")
          .withColumn("registrado_em", F.current_timestamp())
          .write.mode("append").saveAsTable(f"{SCHEMA}.controle_contagens"))

In [0]:
# PROVA DO INCREMENTAL — na 1ª vez refaz a Bronze do zero em 3 etapas;
# depois disso, esta célula só roda a carga incremental normal (arquivos novos).
ESPERA, LOTE2 = "/Volumes/workspace/gs/espera", "2026-11-16"

if not spark.catalog.tableExists(f"{SCHEMA}.controle_contagens"):
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.espera")
    for t in ["bronze_sp156", "bronze_clima", "bronze_ibge", "prata_sp156"]:   # começa do zero
        spark.sql(f"DROP TABLE IF EXISTS {SCHEMA}.{t}")
    dbutils.fs.rm(POUSO_SP156, True)

    for f in dbutils.fs.ls(BASE):                  # lote 2 ainda não chegou
        if LOTE2 in f.name:
            dbutils.fs.mv(f.path, f"{ESPERA}/{f.name}")
    carregar_bronze(); registrar("1 - so lote 1")

    for f in dbutils.fs.ls(ESPERA):                # chega o lote 2
        dbutils.fs.mv(f.path, f"{BASE}/{f.name}")
    carregar_bronze(); registrar("2 - lote 2 enviado")

    carregar_bronze(); registrar("3 - reexecucao sem arquivo novo")
else:
    carregar_bronze()                              # carga incremental do dia a dia

display(spark.sql(f"""SELECT etapa, tabela, linhas, arquivos
  FROM {SCHEMA}.controle_contagens ORDER BY tabela, etapa"""))

xlsx convertidos agora: 0
bronze_sp156 968464
bronze_clima 2
bronze_ibge 1290


etapa,tabela,linhas,arquivos
1 - so lote 1,bronze_clima,1,1
2 - lote 2 enviado,bronze_clima,2,2
3 - reexecucao sem arquivo novo,bronze_clima,2,2
1 - so lote 1,bronze_ibge,645,1
2 - lote 2 enviado,bronze_ibge,1290,2
3 - reexecucao sem arquivo novo,bronze_ibge,1290,2
1 - so lote 1,bronze_sp156,484232,1
2 - lote 2 enviado,bronze_sp156,968464,2
3 - reexecucao sem arquivo novo,bronze_sp156,968464,2


## Prata — tipos, nulos/inválidos, chave sem colisão, 3 regras que barram, MERGE
O SP156 não tem protocolo e a data vem sem hora: linhas idênticas no mesmo arquivo são chamados diferentes. A chave é o hash de todas as colunas + o nº da ocorrência dentro do arquivo — o mesmo arquivo reenviado gera as mesmas chaves, então o MERGE não duplica. Linhas reprovadas vão para `prata_sp156_rejeitados` com o motivo; a contagem por regra fica em `controle_qualidade`.

In [0]:
# PRATA — nulos, chave sem colisão, tipos, inválidos, 3 regras que barram, MERGE
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

b = spark.table(f"{SCHEMA}.bronze_sp156")
NEGOCIO = [c for c in b.columns if c not in ("arquivo_origem", "carregado_em", "_rescued_data")]

# 1) nulos: vazio / "nan" -> null
lixo = ["", "nan", "NaN", "None", "NaT"]
s = b.select(*[F.when(F.trim(F.col(c)).isin(lixo), None).otherwise(F.trim(F.col(c))).alias(c)
               for c in NEGOCIO], "arquivo_origem")

# 2) chave: hash de todas as colunas + nº da ocorrência dentro do arquivo
w = Window.partitionBy("arquivo_origem", *NEGOCIO).orderBy(F.lit(1))
s = (s.withColumn("ocorrencia", F.row_number().over(w))
      .withColumn("id_chamado", F.sha2(F.concat_ws("|",
          *[F.coalesce(F.col(c), F.lit("")) for c in NEGOCIO],
          F.col("ocorrencia").cast("string")), 256))
      .dropDuplicates(["id_chamado"]))          # mesmo chamado em 2 lotes = 1 linha

# 3) tipos corretos
cep = F.regexp_replace(F.regexp_replace("cep", r"\.0$", ""), "[^0-9]", "")
s = (s
  .withColumn("data_de_abertura",    F.expr("try_to_timestamp(data_de_abertura)"))
  .withColumn("data_de_finalizacao", F.expr("try_to_timestamp(data_de_finalizacao)"))
  .withColumn("latitude",  F.expr("try_cast(latitude AS double)"))
  .withColumn("longitude", F.expr("try_cast(longitude AS double)"))
  .withColumn("numero",    F.expr("try_cast(try_cast(numero AS double) AS int)"))
  .withColumn("cep", F.when(F.length(cep) > 0, F.lpad(cep, 8, "0")))
  .withColumn("escopo_vesper", F.col("servico").isin(ESCOPO_SERVICOS)))

# 4) inválido tratado sem barrar: coordenada fora de SP -> null (e conta)
dentro_sp = F.col("latitude").between(-24.1, -23.3) & F.col("longitude").between(-47.0, -46.3)
s = (s.withColumn("coord_fora_sp", F.coalesce(F.col("latitude").isNotNull() & ~dentro_sp, F.lit(False)))
      .withColumn("latitude",  F.when(dentro_sp, F.col("latitude")))
      .withColumn("longitude", F.when(dentro_sp, F.col("longitude"))))

# 5) as 3 regras que barram
R = {
  "R1_data_abertura_valida": F.col("data_de_abertura").isNotNull()
        & (F.col("data_de_abertura") <= F.current_timestamp())
        & (F.col("data_de_abertura") >= F.lit("2020-01-01").cast("timestamp")),
  "R2_fim_depois_do_inicio": F.col("data_de_finalizacao").isNull()
        | (F.col("data_de_finalizacao") >= F.col("data_de_abertura")),
  "R3_distrito_preenchido": F.col("distrito").isNotNull(),
}
for nome, regra in R.items():
    s = s.withColumn(nome, F.coalesce(regra, F.lit(False)))

# materializa uma vez (serverless não tem cache)
s.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}._prata_stage")
st = spark.table(f"{SCHEMA}._prata_stage")

passou = F.col("R1_data_abertura_valida") & F.col("R2_fim_depois_do_inicio") & F.col("R3_distrito_preenchido")
total = st.count()

# 6) contagem por regra -> controle_qualidade
linhas = [(n, total, st.filter(~F.col(n)).count()) for n in R]
linhas.append(("TRATADO_coord_fora_sp_virou_null", total, st.filter("coord_fora_sp").count()))
linhas.append(("TOTAL_barrado_por_alguma_regra", total, st.filter(~passou).count()))
(spark.createDataFrame(linhas, "regra string, avaliadas long, barradas long")
      .withColumn("pct_barrado", F.round(100 * F.col("barradas") / F.col("avaliadas"), 2))
      .withColumn("medido_em", F.current_timestamp())
      .write.mode("overwrite").saveAsTable(f"{SCHEMA}.controle_qualidade"))

# 7) rejeitados (com o motivo) e aprovados (MERGE pela chave)
motivo = F.concat_ws(",", *[F.when(~F.col(n), F.lit(n)) for n in R])
(st.filter(~passou).withColumn("motivo", motivo)
   .write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{SCHEMA}.prata_sp156_rejeitados"))

COLS = ["id_chamado", *NEGOCIO, "escopo_vesper", "arquivo_origem"]
novo = st.filter(passou).select(*COLS).withColumn("processado_em", F.current_timestamp())
if not spark.catalog.tableExists(f"{SCHEMA}.prata_sp156"):
    novo.limit(0).write.saveAsTable(f"{SCHEMA}.prata_sp156")
antes = spark.table(f"{SCHEMA}.prata_sp156").count()
(DeltaTable.forName(spark, f"{SCHEMA}.prata_sp156").alias("t")
   .merge(novo.alias("s"), "t.id_chamado = s.id_chamado")
   .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
depois = spark.table(f"{SCHEMA}.prata_sp156").count()

print(f"chamados distintos: {total} | prata antes do MERGE: {antes} | depois: {depois}")
display(spark.table(f"{SCHEMA}.controle_qualidade"))

chamados distintos: 484232 | prata antes do MERGE: 302923 | depois: 302923


regra,avaliadas,barradas,pct_barrado,medido_em
R1_data_abertura_valida,484232,0,0.0,2026-11-17T04:06:32.941Z
R2_fim_depois_do_inicio,484232,0,0.0,2026-11-17T04:06:32.941Z
R3_distrito_preenchido,484232,181309,37.44,2026-11-17T04:06:32.941Z
TRATADO_coord_fora_sp_virou_null,484232,744,0.15,2026-11-17T04:06:32.941Z
TOTAL_barrado_por_alguma_regra,484232,181309,37.44,2026-11-17T04:06:32.941Z


## Ouro — modelo estrela (fato + 2 dimensões) só com o escopo Vésper
`fato_chamado` (grão = 1 chamado de poste/fiação) · `dim_local` (distrito, zona Centro/Periferia, população IBGE 2022) · `dim_tempo`. LGPD: o endereço só entra na Ouro como `endereco_hash` (SHA-256); a tabela que o painel consome (`ouro_mvp_mapa`) é agregada por distrito.

In [0]:
# ref_populacao_distrito — populacao por distrito (IBGE Censo 2022, agregado 9923)
import requests, unicodedata
from collections import defaultdict
from pyspark.sql import functions as F

url = ("https://servicodados.ibge.gov.br/api/v3/agregados/9923"
       "/periodos/2022/variaveis?localidades=N10[N6[3550308]]")
dados = requests.get(url, timeout=120).json()
var = dados[0]

tem_total, total_por_distrito, soma_por_distrito = False, {}, defaultdict(int)
for res in var["resultados"]:
    cats = res.get("classificacoes", [])
    cat_nome = list(cats[0]["categoria"].values())[0] if cats else "Total"
    for sr in res["series"]:
        nome = sr["localidade"]["nome"]
        raw  = sr["serie"].get("2022")
        val  = int(raw) if raw not in (None, "...", "-", "") else 0
        if cat_nome.strip().lower() == "total":
            tem_total = True; total_por_distrito[nome] = val
        else:
            soma_por_distrito[nome] += val
final = total_por_distrito if tem_total else dict(soma_por_distrito)

def norm(x):
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    return x.strip().lower()

linhas = [(nome, pop, norm(nome)) for nome, pop in final.items()]
ref = spark.createDataFrame(linhas, ["distrito_ibge", "populacao", "distrito_key"])
ref.write.mode("overwrite").saveAsTable("workspace.gs.ref_populacao_distrito")
print("distritos:", ref.count(),
      "| pop total cidade:", ref.agg(F.sum("populacao")).collect()[0][0])

distritos: 96 | pop total cidade: 11451999


In [0]:
# dim_local — distrito + zona (Centro/Periferia) + populacao
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
import unicodedata

def norm(x):
    if x is None: return None
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    return x.strip().lower()
norm_udf = F.udf(norm)

# Centro = Subprefeitura da Se (8 distritos centrais); resto = Periferia
centro = ["se","republica","bela vista","bom retiro","cambuci",
          "consolacao","liberdade","santa cecilia"]

ref = (spark.table("workspace.gs.ref_populacao_distrito")
       .withColumn("distrito_key", norm_udf(F.split(F.col("distrito_ibge"), " - ").getItem(0)))
       .select("distrito_key",
               F.split(F.col("distrito_ibge"), " - ").getItem(0).alias("distrito"),
               F.col("populacao").cast("long").alias("populacao")))

dim_local = (ref
    .withColumn("zona", F.when(F.col("distrito_key").isin(centro), "Centro").otherwise("Periferia"))
    .withColumn("local_id", F.sha2(F.col("distrito_key"), 256))
    .select("local_id","distrito_key","distrito","zona","populacao"))

sch = StructType([
    StructField("local_id", StringType()), StructField("distrito_key", StringType()),
    StructField("distrito", StringType()), StructField("zona", StringType()),
    StructField("populacao", LongType())])
sentinela = spark.createDataFrame(
    [("NAO_INFORMADO","__nao_informado__","Não informado","Não informado", None)], sch)

dim_local = dim_local.unionByName(sentinela)
dim_local.write.mode("overwrite").saveAsTable("workspace.gs.dim_local")
print("linhas dim_local:", dim_local.count())
dim_local.groupBy("zona").agg(F.count("*").alias("distritos"),
                              F.sum("populacao").alias("populacao")).orderBy("zona").show()

linhas dim_local: 97
+-------------+---------+---------+
|         zona|distritos|populacao|
+-------------+---------+---------+
|       Centro|        8|   423536|
|Não informado|        1|     NULL|
|    Periferia|       88| 11028463|
+-------------+---------+---------+



In [0]:
# OURO — dim_tempo + fato_chamado (só escopo Vésper; endereço só como hash)
from pyspark.sql import functions as F
import unicodedata

def norm(x):
    if x is None: return None
    return unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode().strip().lower()
norm_udf = F.udf(norm)

prata = spark.table(f"{SCHEMA}.prata_sp156")
meses = ["jan","fev","mar","abr","mai","jun","jul","ago","set","out","nov","dez"]
dias  = ["dom","seg","ter","qua","qui","sex","sab"]
dim_tempo = (prata.select(F.to_date("data_de_abertura").alias("data")).distinct()
    .withColumn("id_tempo",   F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano",        F.year("data"))
    .withColumn("trimestre",  F.quarter("data"))
    .withColumn("mes",        F.month("data"))
    .withColumn("nome_mes",   F.element_at(F.array(*[F.lit(m) for m in meses]), F.month("data")))
    .withColumn("dia",        F.dayofmonth("data"))
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(d) for d in dias]), F.dayofweek("data"))))
dim_tempo.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.dim_tempo")

p  = prata.filter("escopo_vesper")
dl = spark.table(f"{SCHEMA}.dim_local").select("local_id", "distrito_key")
f = (p.withColumn("distrito_key", norm_udf("distrito"))
      .join(dl, "distrito_key", "left")
      .withColumn("local_id", F.coalesce("local_id", F.lit("NAO_INFORMADO")))
      .withColumn("id_tempo", F.date_format(F.to_date("data_de_abertura"), "yyyyMMdd").cast("int"))
      .withColumn("tempo_conserto_dias",
          F.datediff(F.to_date("data_de_finalizacao"), F.to_date("data_de_abertura")))
      .withColumn("flag_finalizado", F.when(F.lower("status").contains("finaliz")
                                            | F.col("data_de_finalizacao").isNotNull(), 1).otherwise(0))
      .withColumn("flag_valido", F.when(F.col("local_id") != "NAO_INFORMADO", 1).otherwise(0))
      .withColumn("endereco_hash", F.sha2(F.concat_ws("|",                 # LGPD
          F.coalesce("logradouro", F.lit("")),
          F.coalesce(F.col("numero").cast("string"), F.lit("")),
          F.coalesce("cep", F.lit(""))), 256)))
fato = f.select("id_chamado", "id_tempo", "local_id", "servico", "status",
                "data_de_abertura", "data_de_finalizacao", "tempo_conserto_dias",
                "flag_finalizado", "flag_valido", "endereco_hash")
fato.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.fato_chamado")
print("dim_tempo:", dim_tempo.count(), "| fato_chamado (escopo):", fato.count(),
      "| válidos:", fato.filter("flag_valido = 1").count())

dim_tempo: 92 | fato_chamado (escopo): 1113 | válidos: 1113


### As 5 consultas SQL (cada uma imprime a sua Leitura)

In [0]:
# Q1 — Onde a cidade está no escuro: chamados de poste/fiação por distrito
q1 = spark.sql(f"""
SELECT l.distrito, l.zona, COUNT(*) AS chamados,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM {SCHEMA}.fato_chamado f JOIN {SCHEMA}.dim_local l USING (local_id)
WHERE f.flag_valido = 1
GROUP BY l.distrito, l.zona ORDER BY chamados DESC LIMIT 10""")
display(q1); r = q1.collect()
print(f"Leitura: {r[0].distrito} ({r[0].zona}) lidera com {r[0].chamados} chamados ({r[0].pct}%); "
      f"os 10 distritos do topo concentram {sum(x.pct for x in r):.1f}% dos chamados de poste e fiação.")

distrito,zona,chamados,pct
Jabaquara,Periferia,34,3.1
Penha,Periferia,32,2.9
Cursino,Periferia,31,2.8
Vila Prudente,Periferia,29,2.6
Saúde,Periferia,28,2.5
Santana,Periferia,27,2.4
Grajaú,Periferia,27,2.4
Vila Maria,Periferia,25,2.2
Santo Amaro,Periferia,25,2.2
Capão Redondo,Periferia,24,2.2


Leitura: Jabaquara (Periferia) lidera com 34 chamados (3.1%); os 10 distritos do topo concentram 25.3% dos chamados de poste e fiação.


In [0]:
# Q2 — Quanto tempo a falha leva até o fechamento do chamado
q2 = spark.sql(f"""
SELECT ROUND(AVG(tempo_conserto_dias), 1) AS media_dias,
       percentile(tempo_conserto_dias, 0.5) AS mediana_dias,
       percentile(tempo_conserto_dias, 0.9) AS p90_dias,
       COUNT(*) AS finalizados
FROM {SCHEMA}.fato_chamado
WHERE flag_finalizado = 1 AND tempo_conserto_dias >= 0""")
display(q2); r = q2.first()
print(f"Leitura: metade dos {r.finalizados} chamados fecha em até {r.mediana_dias:.0f} dia(s); a média é "
      f"{r.media_dias} dias e 10% passam de {r.p90_dias:.0f} dias. Mede o fechamento no SP156, não o reparo em campo.")

media_dias,mediana_dias,p90_dias,finalizados
18.1,11.0,41.0,119


Leitura: metade dos 119 chamados fecha em até 11 dia(s); a média é 18.1 dias e 10% passam de 41 dias. Mede o fechamento no SP156, não o reparo em campo.


In [0]:
# Q3 — Evolução de chamados por mês
q3 = spark.sql(f"""
SELECT t.ano, t.mes, t.nome_mes, COUNT(*) AS chamados
FROM {SCHEMA}.fato_chamado f JOIN {SCHEMA}.dim_tempo t USING (id_tempo)
GROUP BY t.ano, t.mes, t.nome_mes ORDER BY t.ano, t.mes""")
display(q3); r = q3.collect()
var = 100.0 * (r[-1].chamados - r[0].chamados) / r[0].chamados
print(f"Leitura: de {r[0].nome_mes} ({r[0].chamados}) para {r[-1].nome_mes} ({r[-1].chamados}) "
      f"os chamados variaram {var:+.1f}%.")

ano,mes,nome_mes,chamados
2026,7,jul,315
2026,8,ago,354
2026,9,set,444


Leitura: de jul (315) para set (444) os chamados variaram +41.0%.


In [0]:
# Q4 — % de chamados válidos vs descartados (base SP156 inteira, e quanto disso é escopo Vésper)
q4 = spark.sql(f"""
WITH base AS (
  SELECT 'valido' AS tipo, escopo_vesper FROM {SCHEMA}.prata_sp156
  UNION ALL
  SELECT concat('descartado: ', motivo), escopo_vesper FROM {SCHEMA}.prata_sp156_rejeitados)
SELECT tipo, COUNT(*) AS chamados,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
       SUM(CAST(escopo_vesper AS INT)) AS no_escopo_vesper
FROM base GROUP BY tipo ORDER BY chamados DESC""")
display(q4); r = q4.collect()
val = next(x for x in r if x.tipo == "valido")
desc = [x for x in r if x.tipo != "valido"]
print(f"Leitura: {val.pct}% dos chamados do SP156 passam nas 3 regras da Prata; "
      f"{sum(x.pct for x in desc):.1f}% são descartados ({', '.join(f'{x.tipo[12:]}: {x.chamados}' for x in desc)}). "
      f"No escopo poste/fiação, {val.no_escopo_vesper} chamados válidos e "
      f"{sum(x.no_escopo_vesper or 0 for x in desc)} descartados.")

tipo,chamados,pct,no_escopo_vesper
valido,302923,62.6,1113
descartado: R3_distrito_preenchido,181309,37.4,0


Leitura: 62.6% dos chamados do SP156 passam nas 3 regras da Prata; 37.4% são descartados (R3_distrito_preenchido: 181309). No escopo poste/fiação, 1113 chamados válidos e 0 descartados.


In [0]:
# Q5 (IMPACTO) — chamados por mil habitantes, Centro x Periferia (fato + IBGE)
q5 = spark.sql(f"""
WITH pop AS (SELECT zona, SUM(populacao) AS populacao FROM {SCHEMA}.dim_local
             WHERE zona <> 'Não informado' GROUP BY zona),
     ch  AS (SELECT l.zona, COUNT(*) AS chamados
             FROM {SCHEMA}.fato_chamado f JOIN {SCHEMA}.dim_local l USING (local_id)
             WHERE f.flag_valido = 1 GROUP BY l.zona)
SELECT ch.zona, ch.chamados, pop.populacao,
       ROUND(ch.chamados * 1000.0   / pop.populacao, 3) AS chamados_por_mil_hab,
       ROUND(ch.chamados * 100000.0 / pop.populacao, 1) AS chamados_por_100mil_hab
FROM ch JOIN pop USING (zona) ORDER BY chamados_por_mil_hab DESC""")
display(q5); r = {x.zona: x for x in q5.collect()}
c, p_ = r["Centro"].chamados_por_100mil_hab, r["Periferia"].chamados_por_100mil_hab
print(f"Leitura: o Centro registra {c} chamados de poste/fiação por 100 mil habitantes e a Periferia {p_} "
      f"({c / p_:.1f}x). O denominador é a população residente (IBGE 2022); o Centro recebe muita gente de fora, "
      f"então parte da diferença é circulação, não só infraestrutura.")

zona,chamados,populacao,chamados_por_mil_hab,chamados_por_100mil_hab
Centro,69,423536,0.163,16.3
Periferia,1044,11028463,0.095,9.5


Leitura: o Centro registra 16.3 chamados de poste/fiação por 100 mil habitantes e a Periferia 9.5 (1.7x). O denominador é a população residente (IBGE 2022); o Centro recebe muita gente de fora, então parte da diferença é circulação, não só infraestrutura.


## LGPD e contrato com o produto
O SP156 não traz nome, telefone ou e-mail do solicitante; o quase-identificador (endereço) entra na Ouro só como `endereco_hash`. O painel do Vésper consome a **`ouro_mvp_mapa`**: agregada por distrito, sem nenhum dado individual. Contrato: colunas `distrito, zona, populacao, chamados, chamados_por_mil_hab, chamados_por_100mil_hab, tempo_medio_dias, pct_finalizado, lat, lon` · JSON com 1 objeto por distrito (exemplo em `amostras/ouro_mvp_mapa.json`).

In [0]:
# OURO_MVP_MAPA — contrato com o produto (agregado por distrito) + JSON de exemplo
from pyspark.sql import functions as F
centroide = (spark.table(f"{SCHEMA}.prata_sp156").filter("escopo_vesper")
    .withColumn("distrito_key", norm_udf("distrito"))
    .groupBy("distrito_key")
    .agg(F.round(F.avg("latitude"), 4).alias("lat"), F.round(F.avg("longitude"), 4).alias("lon")))

mvp = (spark.sql(f"""
SELECT l.distrito_key, l.distrito, l.zona, l.populacao,
       COUNT(*) AS chamados,
       ROUND(COUNT(*) * 1000.0   / l.populacao, 3) AS chamados_por_mil_hab,
       ROUND(COUNT(*) * 100000.0 / l.populacao, 1) AS chamados_por_100mil_hab,
       ROUND(AVG(CASE WHEN f.tempo_conserto_dias >= 0 THEN f.tempo_conserto_dias END), 1) AS tempo_medio_dias,
       ROUND(100.0 * SUM(f.flag_finalizado) / COUNT(*), 1) AS pct_finalizado
FROM {SCHEMA}.fato_chamado f JOIN {SCHEMA}.dim_local l USING (local_id)
WHERE f.flag_valido = 1
GROUP BY l.distrito_key, l.distrito, l.zona, l.populacao""")
    .join(centroide, "distrito_key", "left").drop("distrito_key"))
mvp.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.ouro_mvp_mapa")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.saida")
mvp.toPandas().to_json("/Volumes/workspace/gs/saida/ouro_mvp_mapa.json",
                       orient="records", force_ascii=False, indent=1)
print("ouro_mvp_mapa:", mvp.count(), "distritos")
display(mvp.orderBy(F.desc("chamados_por_100mil_hab")))

ouro_mvp_mapa: 95 distritos


distrito,zona,populacao,chamados,chamados_por_mil_hab,chamados_por_100mil_hab,tempo_medio_dias,pct_finalizado,lat,lon
Butantã,Periferia,51715,23,0.445,44.5,26.5,8.7,-23.5797,-46.7295
Liberdade,Centro,66056,20,0.303,30.3,null,0.0,-23.5662,-46.6344
Cursino,Periferia,103171,31,0.300,30.0,19.3,25.8,-23.6166,-46.6191
Morumbi,Periferia,43690,13,0.298,29.8,null,0.0,-23.5967,-46.7141
Santo Amaro,Periferia,85349,25,0.293,29.3,null,0.0,-23.6432,-46.6987
Vila Prudente,Periferia,105690,29,0.274,27.4,22.2,96.6,-23.5915,-46.5769
Vila Guilherme,Periferia,52587,14,0.266,26.6,3.0,14.3,-23.5047,-46.6044
Socorro,Periferia,38051,10,0.263,26.3,null,0.0,-23.6829,-46.7091
Penha,Periferia,132452,32,0.242,24.2,null,0.0,-23.5242,-46.519
Santana,Periferia,115689,27,0.233,23.3,0.0,3.7,-23.495,-46.6267
